# COVID-19 mortality risk: run the study on the official Mexican data

**No GPU needed.** The default CPU runtime is fine.

Then *Runtime → Run all*. It takes about **1 to 1.5 hours**, mostly downloading and reading the government files.

- Progress is saved to Google Drive (`MyDrive/covid_risk`). **If Colab disconnects, run all cells again**: finished steps are skipped.
- At the end, `results_bundle.zip` downloads. It holds only summary results and charts, no patient records.

## 1. Check the machine

In [1]:
!python --version
!free -g | head -2
!df -h /content | tail -1

Python 3.13.16
               total        used        free      shared  buff/cache   available
Mem:              12           0           8           0           3          11
overlay         108G   21G   88G  20% /


## 2. Connect Google Drive (optional: lets an interrupted run resume)

In [2]:
import os
WORKDIR = '/content/covid_risk'            # fallback: local disk (lost if the session ends)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    WORKDIR = '/content/drive/MyDrive/covid_risk'
    print('Google Drive connected - progress will be saved and can resume.')
except Exception as e:
    print('Google Drive not connected (', type(e).__name__, ') - continuing without it.')
os.makedirs(WORKDIR, exist_ok=True)
RAW = f'{WORKDIR}/official_raw'            # the two government zip files are kept here
os.makedirs(RAW, exist_ok=True)
print('Work folder:', WORKDIR)

Mounted at /content/drive
Google Drive connected - progress will be saved and can resume.
Work folder: /content/drive/MyDrive/covid_risk


## 3. Get the code

In [3]:
%cd /content
!rm -rf COVID19-Mortality-Risk
!git clone -q https://github.com/Muneera-DSA/COVID19-Mortality-Risk.git
%cd /content/COVID19-Mortality-Risk
!pip install -q -e ".[explain]"


/content
/content/COVID19-Mortality-Risk
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for covid-risk (pyproject.toml) ... done


## 4. Run everything (about 1 to 1.5 hours)

This downloads `COVID19MEXICO2020.zip` and `COVID19MEXICO2021.zip` from the Ministry of Health (datosabiertos.salud.gob.mx), builds the cohort, and runs every experiment.

**If the download fails:** download the two files yourself from the [official historical-data page](https://www.gob.mx/salud/documentos/datos-abiertos-bases-historicas-direccion-general-de-epidemiologia). Put them in Google Drive in `MyDrive/covid_risk/official_raw`, then run this cell again.

In [4]:
!python -m covid_risk.pipeline --workdir "{WORKDIR}" --raw-dir "{RAW}" --stages data eda benchmark select temporal iecv leakage subgroups explain report bundle

  saved /content/drive/MyDrive/covid_risk/official_raw/COVID19MEXICO2020.zip (61 MB)
  saved /content/drive/MyDrive/covid_risk/official_raw/COVID19MEXICO2021.zip (138 MB)
  COVID19MEXICO2020.zip: 3,868,396 rows read, 1,563,135 confirmed
  COVID19MEXICO2021.zip: 8,830,345 rows read, 2,526,649 confirmed
  12,698,741  records in the yearly files
   4,089,784  confirmed or committee-classified (codes 1-3)
   4,089,784  unique patients (latest version of each record)
   4,074,247  confirmed COVID-19 (codes 1, 3); code 2 excluded (assigned only to deaths)
   4,073,940  valid age, sex and attendance date
   3,922,608  attended 2020-03-01..2021-11-30 with >= 30 days follow-up
   3,922,386  death not dated before attendance
development: 1,511,556 patients (9.4% died); test: 2,410,830 (5.7%)
[eda] finished in 9s
[benchmark] Prior only (no information)      ROC-AUC 0.5000 PR-AUC 0.0942 (0s)
[benchmark] Logistic regression              ROC-AUC 0.9265 PR-AUC 0.5499 (2s)
[benchmark] LDA             

## 5. Download the results and send `results_bundle.zip` back

In [5]:
import os
from google.colab import files
bundle = f'{WORKDIR}/results_bundle.zip'
if os.path.exists(bundle):
    files.download(bundle)
else:
    print('No results_bundle.zip yet: the run above did not finish.')
    print('Scroll up in the run cell and copy the lines containing "Error" or "Traceback".')

---
*Optional quick check (about 10 minutes):* replace step 4 with
`!python -m covid_risk.pipeline --workdir "{WORKDIR}_quick" --raw-dir "{RAW}" --stages data eda benchmark select temporal iecv leakage subgroups explain report bundle --quick`